# 04. Замыкание разрывов

| вход | выход |
|---|---|
| `clean_mask` из этапа 03 | замкнутая маска `closed_mask` |

Контур детали остается разорванным там, где его пересекала размерная линия или накрывала надпись.
Разрыв ломает топологию: кусок контура получает свободный конец и перестает отличаться от обрывка
выносной линии. Поэтому связность чинится до того, как по графу принимаются решения.

Ядро линейное и применяется в нескольких ориентациях: разрыв вдоль линии закрывается ядром,
совпадающим с направлением линии, а поперек линия не растет. Длина ограничена снизу длиной разрыва,
сверху минимальным зазором между соседними линиями.

Главный индикатор перебора это число замкнутых пустот: когда ядро начинает сшивать соседние штрихи
штриховки, каждая ячейка между ними становится пустотой и счетчик прыгает.

In [2]:
from pathlib import Path

from artifact_cache import ArtifactCache
from ipywidgets import Dropdown, IntSlider, interactive
from stage_metrics import MaskMetrics, measure_graph_metrics, measure_mask_metrics
from visualization_old import build_view_window, draw_branch_types, draw_closing_result, window_sliders

from draftslice.common_types import Mask, Mat
from draftslice.image_preparation import load_bgr_image, scale_image
from draftslice.morphology_closing import close_along_orientations, suggest_kernel_length
from draftslice.stroke_graph import build_stroke_graph

DATASET_FOLDER = Path("../data")

## Вход из этапа 03

In [3]:
cache = ArtifactCache()
source_files: list[Path] = [
    path for path in sorted(DATASET_FOLDER.glob("*.jpg")) if cache.has_stage(path, "clean_mask")
]
print(f"файлов с посчитанной чистой маской: {len(source_files)}")


def run_source(file_index: int) -> Path:
    """Выбор файла среди посчитанных на этапе 03."""
    source_path = source_files[file_index]
    stored = cache.load_latest(source_path, "clean_mask")
    print(f"артефакт: {stored.path.name}")
    print(f"параметры этапа 03: {stored.parameters}")
    return source_path


ui_source = interactive(
    run_source,
    file_index=Dropdown(
        options=[(f"[{index}] {path.stem}", index) for index, path in enumerate(source_files)],
        value=0,
        description="файл",
    ),
)
ui_source

файлов с посчитанной чистой маской: 4


interactive(children=(Dropdown(description='файл', options=(('[0] 01', 0), ('[1] 17', 1), ('[2] 19', 2), ('[3]…

In [4]:
source_path: Path = ui_source.result
stored_clean = cache.load_latest(source_path, "clean_mask")
clean_mask: Mask = stored_clean.arrays["clean_mask"]
text_region_mask: Mask = stored_clean.arrays["text_region_mask"]

source_image: Mat = load_bgr_image(source_path)
working_scale: float = clean_mask.shape[1] / source_image.shape[1]
scaled_image: Mat = scale_image(source_image, working_scale)
thickness_class_edges = stored_clean.arrays["thickness_class_edges"]
suggested_length: int = suggest_kernel_length(float(thickness_class_edges[0]))

before_metrics = measure_mask_metrics(clean_mask)
print(f"исходник {source_image.shape[1]}x{source_image.shape[0]} | рабочий масштаб x{working_scale:.1f}")
print(
    f"чернил {before_metrics.ink_share:.2%} | кусков {before_metrics.component_count} | пустот {before_metrics.hole_count}"
)
print(f"граница классов толщины {thickness_class_edges[0]:.2f} px -> ядро {suggested_length} px")

исходник 1085x518 | рабочий масштаб x2.0
чернил 1.57% | кусков 27 | пустот 8
граница классов толщины 3.36 px -> ядро 3 px


## Подбор ядра

In [10]:
def report_closing(before: MaskMetrics, after: MaskMetrics, before_mask: Mask, after_mask: Mask) -> None:
    """Печать того, что сделало замыкание."""
    print(
        f"добавлено {int((after_mask & ~before_mask).sum())} px | чернил {after.ink_share / before.ink_share - 1:+.1%}"
    )
    print(
        f"кусков {before.component_count} -> {after.component_count} | пустот {before.hole_count} -> {after.hole_count}"
    )


def run_closing(kernel_length: int, orientation_count: int) -> Mask:
    """Слайдеры замыкания."""
    closed = close_along_orientations(clean_mask, kernel_length, orientation_count)
    report_closing(before_metrics, measure_mask_metrics(closed), clean_mask, closed)
    draw_closing_result(clean_mask, closed, f"замыкание: линия {kernel_length} px, {orientation_count} ориентаций")
    return closed


ui_closing = interactive(
    run_closing,
    kernel_length=IntSlider(
        value=suggested_length, min=3, max=31, step=2, continuous_update=False, description="длина, px"
    ),
    orientation_count=IntSlider(value=8, min=2, max=16, step=2, continuous_update=False, description="ориентаций"),
)
ui_closing

interactive(children=(IntSlider(value=3, continuous_update=False, description='длина, px', max=31, min=3, step…

## Граф замкнутой маски

Проверка того, что замыкание помогло: контур должен уйти из шпор в ребра узел-узел.

In [11]:
closed_mask: Mask = ui_closing.result
closed_graph = build_stroke_graph(closed_mask, text_region_mask)
metrics = measure_graph_metrics(closed_graph)
print(
    f"ребер {metrics.path_count} | изолятов {metrics.isolated_count} | шпор {metrics.spur_count} | "
    f"узел-узел {metrics.junction_count} | циклов {metrics.cycle_count}"
)
print(f"свободных концов {metrics.free_end_count} | стыков {metrics.branching_node_count}")


def run_graph_view(center_x: float, center_y: float, zoom: float) -> None:
    """Просмотр графа замкнутой маски."""
    window = build_view_window(scaled_image.shape[:2], center_x, center_y, zoom)
    draw_branch_types(closed_graph, scaled_image, window, f"граф замкнутой маски | зум {zoom:.1f}x")


ui_graph_view = interactive(run_graph_view, **window_sliders())
ui_graph_view

ребер 112 | изолятов 22 | шпор 33 | узел-узел 57 | циклов 0
свободных концов 77 | стыков 49


interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='центр X', max=1.0, step=0.0…

## Артефакт для следующего этапа

In [12]:
stage_parameters = {"source_stage": stored_clean.path.name, **ui_closing.kwargs}
saved_path = cache.save(
    source_path,
    "closed_mask",
    stage_parameters,
    closed_mask=closed_mask,
    text_region_mask=text_region_mask,
)
print(f"сохранено: {saved_path}")
print(f"параметры: {stage_parameters}")

сохранено: .cache/17/closed_mask__1c6f5a45c6.npz
параметры: {'source_stage': 'clean_mask__92db17e579.npz', 'kernel_length': 3, 'orientation_count': 8}
